# RuBERT Aspect Category Classification

This notebook fine-tunes `DeepPavlov/rubert-base-cased` for restaurant aspect-category classification.

The task uses seven labels:

- `0`: no aspect category;
- `1`: FOOD;
- `2`: SERVICE;
- `3`: RESTAURANT;
- `4`: AMBIENCE;
- `5`: DRINKS;
- `6`: LOCATION.

Sentences with multiple annotated aspect categories contribute one training example per annotated opinion.


## 1. Setup


In [ ]:
!pip install transformers

import datetime
import random
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from torch.utils.data import (
    DataLoader,
    RandomSampler,
    SequentialSampler,
    TensorDataset,
    random_split,
)
from transformers import (
    AdamW,
    BertForSequenceClassification,
    BertTokenizer,
    get_linear_schedule_with_warmup,
)


In [ ]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"
TRAIN_XML_PATH = "/content/drive/MyDrive/Colab Notebooks/se16_ru_rest_train.xml"
TEST_XML_PATH = "/content/drive/MyDrive/RU_REST_SB1_TEST.xml.B"

CATEGORY_TO_ID = {
    "FOOD": 1,
    "SERVICE": 2,
    "RESTAURANT": 3,
    "AMBIENCE": 4,
    "DRINKS": 5,
    "LOCATION": 6,
}

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
TEST_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
ADAM_EPSILON = 1e-8
EPOCHS = 4
SEED = 42


## 2. Build the single-label dataset

Each annotated opinion contributes one `(sentence, category)` training example. Sentences without opinion annotations receive class `0`.


In [ ]:
def load_single_label_category_dataset(xml_path):
    """Create the sentence/category rows used by the final RuBERT experiment."""
    root = ET.parse(xml_path).getroot()
    rows = []

    for sentence in root.iter("sentence"):
        text_node = sentence.find("text")
        opinions = sentence.find("Opinions")

        if text_node is None:
            continue

        text = (text_node.text or "").strip()

        if opinions is None or len(opinions) == 0:
            rows.append(
                {
                    "text": text,
                    "category": 0,
                }
            )
            continue

        for opinion in opinions.iter("Opinion"):
            category = opinion.attrib["category"].split("#")[0]
            rows.append(
                {
                    "text": text,
                    "category": CATEGORY_TO_ID[category],
                }
            )

    return pd.DataFrame(rows)


train_dataframe = load_single_label_category_dataset(
    TRAIN_XML_PATH
)
test_dataframe = load_single_label_category_dataset(
    TEST_XML_PATH
)

train_dataframe.head()


## 3. Tokenization


In [ ]:
tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)


def encode_texts(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize and pad texts to a fixed maximum sequence length."""
    input_ids = []
    attention_masks = []

    for text in texts:
        encoded = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        input_ids.append(encoded["input_ids"])
        attention_masks.append(encoded["attention_mask"])

    return (
        torch.cat(input_ids, dim=0),
        torch.cat(attention_masks, dim=0),
    )


train_input_ids, train_attention_masks = encode_texts(
    train_dataframe["text"],
    tokenizer,
)
train_labels = torch.tensor(
    train_dataframe["category"].values,
    dtype=torch.long,
)

test_input_ids, test_attention_masks = encode_texts(
    test_dataframe["text"],
    tokenizer,
)
test_labels = torch.tensor(
    test_dataframe["category"].values,
    dtype=torch.long,
)


## 4. Training and validation split

The training dataset is split randomly into 90% training and 10% validation data with seed `42`.


In [ ]:
training_dataset = TensorDataset(
    train_input_ids,
    train_attention_masks,
    train_labels,
)

train_size = int(0.9 * len(training_dataset))
validation_size = len(training_dataset) - train_size

split_generator = torch.Generator().manual_seed(
    SEED
)

train_dataset, validation_dataset = random_split(
    training_dataset,
    [
        train_size,
        validation_size,
    ],
    generator=split_generator,
)

train_dataloader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=TRAIN_BATCH_SIZE,
)

validation_dataloader = DataLoader(
    validation_dataset,
    sampler=SequentialSampler(validation_dataset),
    batch_size=TRAIN_BATCH_SIZE,
)


## 5. RuBERT model and optimizer


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=7,
    output_attentions=False,
    output_hidden_states=False,
)
model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    eps=ADAM_EPSILON,
)

total_training_steps = (
    len(train_dataloader)
    * EPOCHS
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_training_steps,
)


In [ ]:
def classification_accuracy(logits, labels):
    """Compute batch accuracy from class logits."""
    predictions = np.argmax(
        logits,
        axis=1,
    ).flatten()
    labels = labels.flatten()

    return np.mean(
        predictions == labels
    )


def format_elapsed_time(elapsed_seconds):
    """Format elapsed seconds as a timedelta string."""
    return str(
        datetime.timedelta(
            seconds=int(round(elapsed_seconds))
        )
    )


## 6. Fine-tuning


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

training_history = []
training_start_time = time.time()

for epoch_index in range(EPOCHS):
    print(
        f"Epoch {epoch_index + 1} / {EPOCHS}"
    )

    epoch_start_time = time.time()
    total_training_loss = 0.0

    model.train()

    for batch in train_dataloader:
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)

        model.zero_grad()

        outputs = model(
            batch_input_ids,
            token_type_ids=None,
            attention_mask=batch_attention_mask,
            labels=batch_labels,
        )

        loss = outputs.loss
        total_training_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0,
        )

        optimizer.step()
        scheduler.step()

    average_training_loss = (
        total_training_loss
        / len(train_dataloader)
    )
    training_time = format_elapsed_time(
        time.time() - epoch_start_time
    )

    model.eval()

    total_validation_accuracy = 0.0
    total_validation_loss = 0.0
    validation_start_time = time.time()

    for batch in validation_dataloader:
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(
                batch_input_ids,
                token_type_ids=None,
                attention_mask=batch_attention_mask,
                labels=batch_labels,
            )

        total_validation_loss += (
            outputs.loss.item()
        )

        logits = (
            outputs.logits
            .detach()
            .cpu()
            .numpy()
        )
        label_ids = (
            batch_labels
            .detach()
            .cpu()
            .numpy()
        )

        total_validation_accuracy += (
            classification_accuracy(
                logits,
                label_ids,
            )
        )

    average_validation_accuracy = (
        total_validation_accuracy
        / len(validation_dataloader)
    )
    average_validation_loss = (
        total_validation_loss
        / len(validation_dataloader)
    )
    validation_time = format_elapsed_time(
        time.time() - validation_start_time
    )

    training_history.append(
        {
            "epoch": epoch_index + 1,
            "training_loss": average_training_loss,
            "validation_loss": average_validation_loss,
            "validation_accuracy": average_validation_accuracy,
            "training_time": training_time,
            "validation_time": validation_time,
        }
    )

    print(
        f"Training loss: {average_training_loss:.2f}"
    )
    print(
        f"Validation loss: {average_validation_loss:.2f}"
    )
    print(
        "Validation accuracy: "
        f"{average_validation_accuracy:.2f}"
    )

print(
    "Total training time:",
    format_elapsed_time(
        time.time() - training_start_time
    ),
)

training_history_dataframe = pd.DataFrame(
    training_history
).set_index("epoch")

training_history_dataframe


## 7. Evaluation on the SemEval test set


In [ ]:
test_dataset = TensorDataset(
    test_input_ids,
    test_attention_masks,
    test_labels,
)

test_dataloader = DataLoader(
    test_dataset,
    sampler=SequentialSampler(test_dataset),
    batch_size=TEST_BATCH_SIZE,
)

model.eval()

test_logits = []
test_true_labels = []

for batch in test_dataloader:
    batch_input_ids = batch[0].to(device)
    batch_attention_mask = batch[1].to(device)
    batch_labels = batch[2].to(device)

    with torch.no_grad():
        outputs = model(
            batch_input_ids,
            token_type_ids=None,
            attention_mask=batch_attention_mask,
        )

    test_logits.append(
        outputs.logits
        .detach()
        .cpu()
        .numpy()
    )
    test_true_labels.append(
        batch_labels
        .detach()
        .cpu()
        .numpy()
    )

test_logits = np.concatenate(
    test_logits,
    axis=0,
)
test_true_labels = np.concatenate(
    test_true_labels,
    axis=0,
)

test_predictions = np.argmax(
    test_logits,
    axis=1,
)


In [ ]:
test_metrics = {
    "precision": precision_score(
        test_true_labels,
        test_predictions,
        average="weighted",
        zero_division=0,
    ),
    "recall": recall_score(
        test_true_labels,
        test_predictions,
        average="weighted",
        zero_division=0,
    ),
    "f1": f1_score(
        test_true_labels,
        test_predictions,
        average="weighted",
        zero_division=0,
    ),
    "matthews_correlation": matthews_corrcoef(
        test_true_labels,
        test_predictions,
    ),
}

test_metrics
